# R3-N1 WavLM: calibration, bundle, replay and handoff (full run on Colab)
This notebook runs the whole frozen recipe `configs/r3_n1_wavlm_calibration_b.json` from scratch on Colab with **transformers 5.22.0**: the tests, then audit, fit, operating, package, a two-tier replay, an end-to-end CLI check, and the handoff record. It creates a **new** run folder under `runs/r3/r3_n1_wavlm/` (no name to set) and a new bundle; nothing from an earlier run is continued or overwritten.

The recipe and the plan (`WAVLM_CALIBRATION.md`) are frozen: no setting changes between the first execution (local, discarded at the owner's request) and this one. It trains nothing and reads no test or Chatterbox audio.

Sync first (same paths on Drive): the code, tests, recipe and notebook from the sync list; the R3-U1 exports `reports/r3/r3_u1_B/`; the source bundle `bundles/r3_u1_wavlm_head_B/`; `data/manifests/cohort_v1.parquet`, `data/splits/protocol_B_v1.parquet`, `data/eda/r2_s3_shortcut/duration_matched_B_v1.json` and the shards `data/processed/cohort_v1/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Pin transformers
Colab ships a newer patch version. Run this cell, then **restart the session** (Runtime, Restart session) and continue from the next cell. The check cell fails if the version is wrong.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.22.0'], check=True)
import importlib.metadata as md
print('transformers', md.version('transformers'))

In [ ]:
from pathlib import Path
import hashlib, io, json, os, re, subprocess, sys, time
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')  # Change if your project is elsewhere.
OWNER = 'Subhan'
SHARDS = ROOT / 'data/processed/cohort_v1'
BUNDLE = ROOT / 'bundles/r3_n1_wavlm_head_B_calibrated'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, str(ROOT / 'src'))
import torch, transformers, numpy, scipy, pyarrow
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| numpy', numpy.__version__, '| scipy', scipy.__version__, '| pyarrow', pyarrow.__version__)
assert transformers.__version__ == '5.22.0', 'The pin is transformers 5.22.0: install it and restart the session'
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()
required = ['configs/r3_n1_wavlm_calibration_b.json', 'scripts/r3_n1_wavlm_calibrate.py', 'scripts/r3_n1_wavlm_replay.py',
            'src/echoproof/evaluation/r3_n1_wavlm.py', 'src/echoproof/evaluation/r3_n1_wavlm_operating.py', 'src/echoproof/models/r3_n1_wavlm_bundle.py',
            'reports/r3/r3_u1_B/selection.json', 'reports/r3/r3_u1_B/evidence.json', 'bundles/r3_u1_wavlm_head_B/bundle.json',
            'data/manifests/cohort_v1.parquet', 'data/splits/protocol_B_v1.parquet', 'data/eda/r2_s3_shortcut/duration_matched_B_v1.json']
missing = [p for p in required if not (ROOT / p).is_file()]
assert not missing, f'Missing: {missing}'
assert SHARDS.is_dir(), 'Set SHARDS to the canonical shard folder'
assert not BUNDLE.exists(), f'{BUNDLE} already exists: the packager never overwrites. Rename or remove the old bundle on Drive and rerun this cell.'
print('inputs present; the calibrated bundle folder is free')

## Tests (nothing is fitted until they pass)
The CPU tests, the real-library adapter tests (a tiny random WavLM through transformers 5.22.0) and the inference-path tests. A failure or a skip stops the notebook.

In [ ]:
tests = ['tests/test_r3_n1_wavlm_numeric.py', 'tests/test_r3_n1_wavlm_pipeline.py', 'tests/test_r3_n1_wavlm_bundle.py', 'tests/test_r3_n1_wavlm_replay.py',
         'tests/test_r3_u1_adapter.py', 'tests/test_r1_n2_inference.py', 'tests/test_r2_bundle.py']
check = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', '-rs', *tests], text=True, capture_output=True)
print(check.stdout[-3500:], check.stderr[-1500:])
# One skip is by design: test_r1_n2_inference.py tests the fallback for a machine WITHOUT matplotlib, so it is skipped wherever matplotlib is installed.
ALLOWED_SKIPS = ('matplotlib is installed',)
skips = re.findall(r'^SKIPPED \[\d+\] (.+)$', check.stdout, flags=re.M)
unexpected = [x for x in skips if not any(a in x for a in ALLOWED_SKIPS)]
assert check.returncode == 0, 'Resolve the test failures first'
assert not unexpected, f'Unexpected skips (a missing dependency?): {unexpected}'
print('tests passed; skips:', skips or 'none')

## Stage A: audit
Creates the new run folder, freezes a copy of the recipe, verifies the saved exports against the recipe, `evidence.json` and the locked cohort and split manifests. The run folder is parsed from the output.

In [ ]:
def stage(command, *extra):
    result = subprocess.run([sys.executable, 'scripts/r3_n1_wavlm_calibrate.py', command, '--root', str(ROOT), '--owner', OWNER, *extra], text=True, capture_output=True)
    print(result.stdout[-3000:], result.stderr[-2000:])
    assert result.returncode == 0, f'{command} failed'
    return result.stdout
out = stage('audit')
RUN = Path(re.search(r'audit passed: (.+)', out).group(1).strip())
print('RUN =', RUN)

## Stage B: choose and fit the mapping (calibration role only)
Leave-one-speaker-out comparison of identity, temperature and affine, the pre-declared selection rule, the final fit and the freeze. The operating file is not opened.

In [ ]:
stage('fit', '--run-dir', str(RUN))

## Stage C: the operating role, opened once
Acceptance rule, threshold, rows A to D, reliability, speaker diagnostics, bootstrap. A second attempt in the same run is refused.

In [ ]:
stage('operating', '--run-dir', str(RUN))

## Stage E: package the calibrated bundle

In [ ]:
stage('package', '--run-dir', str(RUN))

## Replay (transformers 5.22.0)
All operating recordings and 100 each from the selection and calibration roles, in two tiers (the bundle's scorer on canonical windows, and WAV bytes through `analyze`). Tolerance on recording scores 1e-3; a flip is explained only when the expected score is within the tolerance of the threshold.

In [ ]:
subprocess.run([sys.executable, 'scripts/r3_n1_wavlm_replay.py', '--root', str(ROOT), '--run-dir', str(RUN), '--shard-dir', str(SHARDS), '--owner', OWNER,
                '--device', 'cuda' if torch.cuda.is_available() else 'cpu'], check=True)

## End-to-end CLI check, reproduction check and handoff
`python -m echoproof.cli predict` with no `--threshold` (the bundle's own rule) must equal the library result. The reproduction check compares this run with the reference from the first, discarded local execution; it reports differences and does not gate.

In [ ]:
import numpy as np, soundfile as sf
from echoproof import inference as inf
from echoproof.data import cohort
from echoproof.models.r3_n1_wavlm_bundle import load_wavlm_bundle
from echoproof.training.datasets import ShardWaveformLoader
device = 'cuda' if torch.cuda.is_available() else 'cpu'
replay = json.loads((RUN / 'handoff/replay.json').read_text(encoding='utf-8'))
assert replay['passed'] is True, 'The replay did not pass'
rec = replay['records'][0]
row = {r['recording_id']: r for r in cohort.read_table(str(ROOT / 'data/manifests/cohort_v1.parquet'))}[rec['recording_id']]
loader = ShardWaveformLoader(SHARDS, {rec['recording_id']: row['canonical_path']})
wave = loader(rec['recording_id']); loader.close()
wav_path = Path('/content/replay_sample.wav')
sf.write(str(wav_path), np.rint(wave.astype(np.float64) * 32768).astype('<i2'), 16000, subtype='PCM_16')
cli = subprocess.run([sys.executable, '-m', 'echoproof.cli', 'predict', '--bundle', str(BUNDLE), '--audio', str(wav_path)], text=True, capture_output=True)
print(cli.stdout[-1500:], cli.stderr[-800:])
assert cli.returncode == 0, 'CLI predict failed'
cli_result = json.loads(cli.stdout)
library = inf.predict_recording(wav_path.read_bytes(), load_wavlm_bundle(BUNDLE, device=device))
assert abs(cli_result['score'] - library.score) < 1e-9 and cli_result['assessment'] == library.assessment and cli_result['score_type'] == library.score_type
print('CLI equals library:', cli_result['score'], cli_result['assessment'], cli_result['score_type'])

LOCAL_REFERENCE = {'selected_mapping': 'affine', 'a': 1.1079811437013474, 'b': 0.9165647745687742, 'threshold': 0.4040371283027592, 'accepted': True,
                   'raw_log_loss': 0.1749, 'calibrated_log_loss': 0.1565, 'operating_macro_f1': 0.9129, 'operating_genuine_fpr': 0.1885}
fit = json.loads((RUN / 'fit/calibration.json').read_text(encoding='utf-8'))
op = json.loads((RUN / 'operating/threshold.json').read_text(encoding='utf-8'))
metrics = json.loads((RUN / 'operating/metrics.json').read_text(encoding='utf-8'))
now = {'selected_mapping': fit['selected_mapping'], 'a': fit['selected_parameters'].get('a'), 'b': fit['selected_parameters'].get('b'),
       'threshold': op['decision_rule']['threshold'], 'accepted': op['calibration_accepted'], 'raw_log_loss': round(metrics['probability']['raw']['log_loss'], 4),
       'calibrated_log_loss': round(metrics['probability']['selected']['log_loss'], 4) if 'selected' in metrics['probability'] else None,
       'operating_macro_f1': round(metrics['rows']['D_selected_at_its_optimum']['full']['macro_f1'], 4) if 'D_selected_at_its_optimum' in metrics['rows'] else None,
       'operating_genuine_fpr': round(metrics['rows']['D_selected_at_its_optimum']['full']['genuine_fpr'], 4) if 'D_selected_at_its_optimum' in metrics['rows'] else None}
reproduction = {k: {'local_reference': LOCAL_REFERENCE[k], 'this_run': now[k], 'equal': (abs(LOCAL_REFERENCE[k] - now[k]) < 1e-6) if isinstance(LOCAL_REFERENCE[k], float) and now[k] is not None else LOCAL_REFERENCE[k] == now[k]} for k in LOCAL_REFERENCE}
for k, v in reproduction.items():
    print(f"{k:24s} reference {v['local_reference']!s:22s} this run {v['this_run']!s:22s} {'same' if v['equal'] else 'DIFFERENT'}")

evidence = json.loads((RUN / 'operating/evidence.json').read_text(encoding='utf-8'))
calibration = json.loads((BUNDLE / 'calibration/calibration.json').read_text(encoding='utf-8'))
handoff = {'task': 'R3-N1-WavLM handoff', 'created_utc': time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()), 'owner': OWNER, 'run_folder': str(RUN), 'bundle': str(BUNDLE),
           'bundle_json_sha256': sha256(BUNDLE / 'bundle.json'), 'calibration_component_sha256': sha256(BUNDLE / 'calibration/calibration.json'),
           'source_checkpoint_sha256': calibration['source']['source_checkpoint_sha256'], 'calibration_accepted': calibration['calibration_accepted'], 'score_type': calibration['score_type'],
           'method': calibration['method'], 'decision_rule': calibration['operating_decision_rule'], 'transformers': transformers.__version__, 'torch': torch.__version__,
           'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'replay_report_sha256': sha256(RUN / 'handoff/replay.json'),
           'replay_summary': {k: replay[k] for k in ('recordings', 'tiers', 'tier1_vs_tier2', 'passed') if k in replay},
           'operating_evidence_sha256': sha256(RUN / 'operating/evidence.json'), 'operating_outputs': evidence['outputs'],
           'fit_stage_complete_sha256': sha256(RUN / 'fit/stage_complete.json'), 'input_audit_sha256': sha256(RUN / 'input_audit.json'),
           'cli_check': {'score': cli_result['score'], 'assessment': cli_result['assessment'], 'score_type': cli_result['score_type']},
           'reproduction_of_first_local_execution': reproduction,
           'status': 'development bundle; not promoted, not locked; independent review pending'}
with (RUN / 'handoff/handoff.json').open('x', encoding='utf-8') as f:
    json.dump(handoff, f, indent=2, sort_keys=True)
print('handoff written:', RUN / 'handoff/handoff.json')